In [ ]:
from pytorch_grad_cam import GradCAM, HiResCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, FullGrad
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision.io import read_image
import matplotlib.pyplot as plt
from src.grading_model.grading_model import GradingModel
import torch
import cv2
import numpy as np
import os

In [ ]:
LESION_COLORS = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (100, 255, 50)]

GRADING_MODEL_PATH = ""

In [ ]:
model = GradingModel(num_outputs=1)
model.load_state_dict(torch.load(GRADING_MODEL_PATH, weights_only=True))

In [ ]:
model

In [ ]:
masks_dir = 'datasets/segmentation_dataset/train_set/masks'
mask_dirs = [os.path.join(masks_dir, mask_dir) for mask_dir in sorted(os.listdir(masks_dir)) if mask_dir != 'optic_disc']

masks_paths = [[os.path.join(mask_dir, mask_filename) for mask_filename in sorted(os.listdir(mask_dir))] for mask_dir in mask_dirs]

In [ ]:
images_dir = 'datasets/segmentation_dataset/train_set/images'
images_paths = sorted([os.path.join(images_dir, image_filename) for image_filename in os.listdir(images_dir)])

In [ ]:
target_layers = [model.post_f_low_seq[18]]

# We have to specify the target we want to generate the CAM for.
targets = [BinaryClassifierOutputTarget(1)]

In [ ]:
i = 0
for image_path, masks in zip(images_paths, zip(*masks_paths)): # Double zip so that masks can be set as tuple
  input_tensor = read_image(image_path)

  rgb_img = input_tensor.permute(1, 2, 0).numpy()

  fig = plt.figure(figsize=(20, 10))
  ax = fig.add_subplot(1, 4, 1)
  ax.imshow(rgb_img)
  ax.axis('off')

  ground_truth_mask = np.zeros((640, 640, 3), dtype=np.uint8)
  print(len(masks))
  for mask_index, mask_path in enumerate(masks):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    ground_truth_mask[mask > 0, :] = LESION_COLORS[mask_index]

  normalized_input_tensor = input_tensor.float().unsqueeze(0) / 255
  normalized_rgb_img = rgb_img / 255

  # Construct the CAM object once, and then re-use it on many images.
  with GradCAM(model=model, target_layers=target_layers) as cam:
    # You can also pass aug_smooth=True and eigen_smooth=True, to apply smoothing.
    grayscale_cam = cam(input_tensor=normalized_input_tensor, targets=targets)
    # In this example grayscale_cam has only one image in the batch:
    grayscale_cam = grayscale_cam[0, :]
    visualization = show_cam_on_image(normalized_rgb_img, grayscale_cam, use_rgb=True)
    # You can also get the model outputs without having to redo inference
    model_outputs = cam.outputs
    print(model_outputs[0])
    ax = fig.add_subplot(1, 4, 2)
    ax.imshow(visualization)
    ax.axis('off')

  if model_outputs[0] < 0.5:
    grayscale_cam = np.zeros((640, 640), dtype=np.uint8)

  ax = fig.add_subplot(1, 4, 3)
  ax.imshow(ground_truth_mask)
  ax.axis('off')

  std = grayscale_cam.std()
  thresholded = ((grayscale_cam > 2*std) * 255).astype(np.uint8)
  thresholded = cv2.cvtColor(thresholded, cv2.COLOR_GRAY2BGR)
  merged = cv2.addWeighted(ground_truth_mask, 0.5, thresholded, 0.5, 0)
  ax = fig.add_subplot(1, 4, 4)
  ax.imshow(merged)
  ax.axis('off')
  if i > 10:
    break
  else:
    i += 1

In [ ]:
threshold = 0.4

std = grayscale_cam.std()
thresholded = (grayscale_cam > 3*std).astype(np.uint8) * 255
thresholded = cv2.cvtColor(thresholded, cv2.COLOR_GRAY2BGR)

merged = cv2.addWeighted(rgb_img, threshold, thresholded, 1-threshold, 0)

fig = plt.figure(figsize=(15, 15))
ax = fig.add_subplot(2, 1, 1)
ax.imshow(merged)
ax.axis('off')